# Query Export for Tableau Dashboard

This notebook connects to `airline_delay_analytics` and runs each analytical SQL
query stored in `sql/`, exporting the results to CSV files in `data/exports/`.

Tableau Public doesn't support live database connections (that's a Tableau Desktop
feature) — it only accepts file-based sources like CSV, Excel, or Google Sheets.
Exporting query results here keeps a single source of truth for the SQL logic (the
`.sql` files) while still producing dashboard-ready files.

**Queries exported:**
- `carrier_scorecard` — on-time %, avg delay, and cancellation rate by airline
- `delay_causes_breakdown` — root-cause share of total delay minutes
- `monthly_trends` — seasonal on-time %, delay, and cancellation patterns
- `route_reliability` — least reliable origin-destination pairs (200+ flight minimum)
- `day_of_week_trends` — on-time % and delay by day of week
- `carrier_ranking` — window function (`RANK`) example, ranking carriers by on-time %
- `month_over_month_change` — CTE + window function (`LAG`) example, tracking
  month-over-month change in on-time %

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [2]:
load_dotenv('../.env')

engine = create_engine(
    f'postgresql://{os.getenv("DB_USER")}@{os.getenv("DB_HOST")}:{os.getenv("DB_PORT")}/{os.getenv("DB_NAME")}'
)

## Export 1: Analytical Query Results

Exporting the 7 pre-aggregated analytical queries from `sql/` as individual CSVs.
These are small, summary-level outputs (one row per carrier, per month, etc.) suited
for straightforward chart-building without needing any further joining or aggregation
in Tableau.

In [3]:
queries = {
    'carrier_ranking': open('../sql/carrier_ranking.sql').read(),
    'carrier_scorecard': open('../sql/carrier_scorecard.sql').read(),
    'day_of_week_trends': open('../sql/day_of_week_trends.sql').read(),
    'delay_causes_breakdown': open('../sql/delay_causes_breakdown.sql').read(),
    'monthly_trends': open('../sql/monthly_trends.sql').read(),
    'month_over_month_change': open('../sql/month_over_month_change.sql').read(),
    'route_reliability': open('../sql/route_reliability.sql').read(),
}

for name, sql in queries.items():
    df = pd.read_sql(text(sql), engine)
    df.to_csv(f'../data/exports/queries/{name}.csv', index=False)
    print(f'{name}: {df.shape}')

carrier_ranking: (14, 3)
carrier_scorecard: (14, 5)
day_of_week_trends: (7, 5)
delay_causes_breakdown: (1, 5)
monthly_trends: (12, 5)
month_over_month_change: (12, 4)
route_reliability: (20, 5)


## Export 2: Full Star Schema Tables

Exporting the complete `dim_date`, `dim_airport`, `dim_carrier`, and `fact_flights`
tables as CSVs, so the full star schema (including the 7M-row fact table) can be
loaded into Tableau directly. This allows building new joins, filters, and visuals
within Tableau itself, rather than being limited to the pre-defined analytical
queries above.

The three dimension tables are small and exported via `pandas.read_sql()` like the
analytical queries above. `fact_flights`, at ~7 million rows, is exported separately
using Postgres's native `COPY TO STDOUT` command instead — `read_sql()` round-trips
every value through a Python object before pandas can write it out, which becomes
prohibitively slow at this scale. `COPY` streams directly from Postgres to a CSV file
with no intermediate Python overhead, completing in seconds rather than many minutes.

In [4]:
tables = ['dim_date', 'dim_airport', 'dim_carrier']

for table in tables:
    df = pd.read_sql(text(f'SELECT * FROM {table}'), engine)
    df.to_csv(f'../data/exports/tables/{table}.csv', index=False)
    print(f'{table}: {df.shape}')

dim_date: (365, 7)
dim_airport: (352, 4)
dim_carrier: (14, 2)


In [7]:
import psycopg2

conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'), user=os.getenv('DB_USER'),
    host=os.getenv('DB_HOST'), port=os.getenv('DB_PORT')
)

cur = conn.cursor()

with open('../data/exports/tables/fact_flights.csv', 'w') as f:
    cur.copy_expert('COPY fact_flights TO STDOUT WITH CSV HEADER', f)

conn.close()
print('fact_flights exported via COPY')

fact_flights exported via COPY
